# CSE457 – Xử lý âm thanh và tiếng nói
## LAB 1: Phân tích và xử lý tín hiệu âm thanh số
---
| Thông tin | Nội dung |
|-----------|----------|
| **Sinh viên** | Dương Tiến Đạt |
| **MSSV** | 2251172268 |
| **Học phần** | CSE457 |
| **Notebook** | Lab01_2251172268_DuongTienDat.ipynb |

> Notebook này chạy **hoàn toàn tự động** từ đầu đến cuối.  
> Chọn file âm thanh ở cell **Bước 1.2** bằng cách đổi biến `CHOICE`.


## 0. Cài đặt và import

In [ ]:
# Kiểm tra và cài thư viện nếu chưa có
import subprocess, sys
pkgs = ['numpy','scipy','matplotlib','soundfile','pyttsx3']
for p in pkgs:
    try:
        __import__(p)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])

import os, wave, struct, warnings
import numpy as np
import matplotlib
matplotlib.rcParams.update({'figure.dpi':120,'font.size':9})
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from scipy.signal import (firwin, lfilter, freqz, group_delay,
                           spectrogram as _spectrogram,
                           resample_poly, find_peaks)
from math import gcd
import soundfile as sf
warnings.filterwarnings('ignore')

# Đường dẫn thư mục
BASE_DIR    = os.path.dirname(os.path.abspath('__file__'))
AUDIO_DIR   = os.path.join(BASE_DIR, 'audio')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
os.makedirs(AUDIO_DIR,   exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print('✓ Import OK | BASE_DIR =', BASE_DIR)


## 1. Tạo dữ liệu âm thanh

### 1.1 Các hàm tiện ích

In [ ]:
# ── Lưu WAV ─────────────────────────────────────────────
def save_wav(sig, fs, fname):
    path = os.path.join(AUDIO_DIR, fname)
    s16  = (np.clip(sig,-1,1)*32767).astype(np.int16)
    with wave.open(path,'w') as wf:
        wf.setnchannels(1); wf.setsampwidth(2)
        wf.setframerate(fs)
        wf.writeframes(struct.pack(f'<{len(s16)}h',*s16))
    return path

# ── Đọc WAV → dict ───────────────────────────────────────
def read_wav(path):
    fsize = os.path.getsize(path)
    with wave.open(path,'r') as wf:
        fs=wf.getframerate(); ch=wf.getnchannels()
        sw=wf.getsampwidth(); bits=sw*8
        nfr=wf.getnframes(); raw=wf.readframes(nfr)
    dt  = {8:np.uint8,16:np.int16,32:np.int32}.get(bits,np.int16)
    s   = np.frombuffer(raw,dtype=dt)
    if ch>1: s=s.reshape(-1,ch)
    mv  = 128.0 if bits==8 else float(2**(bits-1))
    x   = (s.astype(np.float64)-(128 if bits==8 else 0))/mv
    x   = np.clip(x,-1,1)
    xm  = np.mean(x,axis=1) if ch>1 else x.flatten()
    return dict(x=x,x_mono=xm,fs=fs,channels=ch,bit_depth=bits,
                n_frames=nfr,duration=nfr/fs,sample_width=sw,
                file_size=fsize,filename=os.path.basename(path))

# ── dBFS ─────────────────────────────────────────────────
def db(v, floor=-120):
    return float(np.maximum(20*np.log10(np.abs(v)+1e-15), floor))

# ── Lưu figure ───────────────────────────────────────────
def savefig(fig, name):
    p = os.path.join(FIGURES_DIR, name)
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'  → {p}')

print('✓ Hàm tiện ích OK')


### 1.2 Tạo tín hiệu âm thanh mẫu

In [ ]:
FS = 22050

# Sóng sine 440 Hz
t = np.linspace(0,5,int(5*FS),endpoint=False)
save_wav(0.8*np.sin(2*np.pi*440*t), FS, 'music_sine_440hz.wav')

# Hợp âm Do trưởng + transient
chord = np.zeros_like(t)
for f0 in [261.63,329.63,392.00]:
    for k in range(1,6):
        if f0*k<FS/2: chord+=(0.4/k)*np.sin(2*np.pi*f0*k*t)
rng=np.random.default_rng(42)
for bt in np.arange(0,5,1.0):
    i0=int(bt*FS); il=min(int(0.08*FS),len(chord)-i0)
    if il>0:
        env=np.exp(-50*np.linspace(0,.08,il))
        chord[i0:i0+il]+=0.3*env*rng.standard_normal(il)
chord=chord/(np.max(np.abs(chord))+1e-9)*0.85
save_wav(chord, FS, 'music_chord_Cmaj.wav')

# Giọng nói tổng hợp – source-filter model cải tiến (F0 contour + jitter)
def make_speech(fs=FS, dur=5.0, gender='male'):
    N=int(dur*fs); sig=np.zeros(N); rng2=np.random.default_rng(42 if gender=='male' else 77)
    f0m,f0r=(120,40) if gender=='male' else (200,60)
    fmts=[(730,90,1090,110,2440,170),(270,80,2290,200,3010,250),
          (520,85,1190,130,2390,180),(390,70,1990,180,2550,200)]
    if gender=='female':
        fmts=[(850,90,1220,120,2810,180),(310,90,2790,250,3310,280),
              (600,90,1170,150,2670,200),(460,80,2070,200,2760,220)]
    tf=np.linspace(0,dur,N,endpoint=False)
    f0c=f0m+f0r*np.sin(2*np.pi*0.3*tf)+f0r*0.3*np.sin(2*np.pi*1.5*tf+1.2)
    f0c=np.clip(f0c,f0m*0.6,f0m*1.8)
    pos=0; si=0
    while pos<N:
        if rng2.random()<0.12 and pos>int(0.3*fs):
            pos+=int(rng2.uniform(0.05,0.20)*fs); continue
        sn=min(int(rng2.uniform(0.08,0.18)*fs),N-pos)
        if sn<=0: break
        f1,bw1,f2,bw2,f3,bw3=fmts[si%len(fmts)]; si+=1
        if rng2.random()<0.15:
            src=rng2.standard_normal(sn)*0.4
        else:
            fl=float(np.mean(f0c[pos:pos+sn])); per=fs/fl
            src=np.zeros(sn); tv=0.0
            while tv<sn:
                idx=int(tv)
                if idx<sn: src[idx]+=rng2.uniform(0.95,1.05)
                tv+=per*(1+rng2.uniform(-0.02,0.02))
        ssig=np.zeros(sn)
        for fc,bw in [(f1,bw1),(f2,bw2),(f3,bw3)]:
            if fc>=fs/2: continue
            r=np.exp(-np.pi*bw/fs); th=2*np.pi*fc/fs
            ssig+=lfilter([1-r],[1,-2*r*np.cos(th),r*r],src)
        fi=min(int(0.015*fs),sn//4); fo=min(int(0.02*fs),sn//4)
        env=np.ones(sn); env[:fi]=np.linspace(0,1,fi); env[sn-fo:]=np.linspace(1,0,fo)
        sig[pos:pos+sn]+=ssig*env; pos+=sn
    pk=np.max(np.abs(sig)); sig=sig/pk*0.82 if pk>0 else sig
    sig+=rng2.standard_normal(N)*0.003
    return np.clip(sig,-1,1)

save_wav(make_speech(gender='male'),   FS, 'speech_male_synth.wav')
save_wav(make_speech(gender='female'), FS, 'speech_female_synth.wav')

# Hội thoại tổng hợp: nam + nữ xen kẽ
segs=[]; sil=np.zeros(int(0.35*FS))
for i in range(6):
    g='male' if i%2==0 else 'female'
    segs.append(make_speech(fs=FS,dur=1.2,gender=g)); segs.append(sil.copy())
dlg=np.concatenate(segs); dlg=dlg/np.max(np.abs(dlg))*0.85
save_wav(dlg, FS, 'speech_dialogue_synthetic.wav')

print('\n✓ File âm thanh đã tạo:')
for f in sorted(os.listdir(AUDIO_DIR)):
    p=os.path.join(AUDIO_DIR,f)
    print(f'  {f:<45} {os.path.getsize(p)//1024:>4}KB')


### 1.3 Chọn file phân tích

**Thay đổi `CHOICE`** để phân tích file khác:
- `1` = `music_chord_Cmaj.wav` (nhạc)
- `2` = `speech_male_synth.wav` (giọng nam)
- `3` = `speech_female_synth.wav` (giọng nữ)
- `4` = `speech_dialogue_synthetic.wav` (hội thoại)


In [ ]:
CHOICE = 1   # ← Đổi số này để chọn file khác

wavs = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')])
print('Danh sách file:')
for i,f in enumerate(wavs,1): print(f'  [{i}] {f}')

if 1 <= CHOICE <= len(wavs):
    SEL_PATH = os.path.join(AUDIO_DIR, wavs[CHOICE-1])
else:
    SEL_PATH = os.path.join(AUDIO_DIR, wavs[0])

info = read_wav(SEL_PATH)
x, fs, dur = info['x_mono'], info['fs'], info['duration']
# Phân loại prefix dựa trên tên file
fname_base = os.path.splitext(info['filename'])[0]
prefix = 'speech' if any(k in fname_base.lower() for k in ('speech','voice','tts')) else 'music'
print(f'\n✓ File đã chọn: {SEL_PATH}')
print(f'  prefix = "{prefix}"')


---
## Phần A – Đọc và kiểm tra dữ liệu âm thanh

In [ ]:
# In metadata đầy đủ
fs_v=info['fs']; ch=info['channels']; bd=info['bit_depth']
print('='*55)
print(f"  File        : {info['filename']}")
print(f"  Duration    : {dur:.4f} s")
print(f"  Fs          : {fs_v:,} Hz")
print(f"  Nyquist     : {fs_v//2:,} Hz")
print(f"  Channels    : {ch}")
print(f"  Bit depth   : {bd} bit/sample")
print(f"  N frames    : {info['n_frames']:,}")
print(f"  PCM bitrate : {fs_v*bd*ch:,} bit/s  ({fs_v*bd*ch/1000:.1f} kbps)")
print(f"  File size   : {info['file_size']:,} bytes  ({info['file_size']/1024/1024:.4f} MB)")
print(f"  Peak        : {np.max(np.abs(x)):.6f}  ({db(np.max(np.abs(x))):.2f} dBFS)")
print(f"  RMS         : {np.sqrt(np.mean(x**2)):.6f}  ({db(np.sqrt(np.mean(x**2))):.2f} dBFS)")
print('='*55)


In [ ]:
# Waveform + chuẩn hóa
t_ax = np.arange(len(x))/fs
peak_v = np.max(np.abs(x)); rms_v = np.sqrt(np.mean(x**2))
x_fs  = x/peak_v if peak_v>0 else x.copy()
x_rms = np.clip(x*(0.1/rms_v),-1,1) if rms_v>0 else x.copy()

fig,axes=plt.subplots(3,1,figsize=(13,8),sharex=True)
for ax,sig,clr,ttl in [
    (axes[0],x,'steelblue',f'Gốc | Peak={db(peak_v):.1f}dBFS  RMS={db(rms_v):.1f}dBFS'),
    (axes[1],x_fs,'darkorange',f'Full-scale norm | Peak=0dBFS  RMS={db(np.sqrt(np.mean(x_fs**2))):.1f}dBFS'),
    (axes[2],x_rms,'seagreen',f'RMS norm (−20dBFS target) | Peak={db(np.max(np.abs(x_rms))):.1f}dBFS')]:
    ax.plot(t_ax,sig,color=clr,lw=0.3,alpha=0.9)
    r=np.sqrt(np.mean(sig**2))
    ax.axhline(r,color='orange',lw=0.9,ls='-.',label=f'RMS={r:.4f}')
    ax.axhline(-r,color='orange',lw=0.9,ls='-.')
    ax.axhline(1,color='red',lw=0.6,ls='--',alpha=0.5)
    ax.axhline(-1,color='red',lw=0.6,ls='--',alpha=0.5)
    ax.set_ylabel('Biên độ',fontsize=8); ax.set_title(ttl,fontsize=9,fontweight='bold')
    ax.legend(fontsize=7,loc='upper right'); ax.set_ylim([-1.15,1.15]); ax.grid(True,alpha=.3)
axes[-1].set_xlabel('Thời gian (s)')
fig.suptitle(f'Phần A – Metadata & Chuẩn hóa: {info["filename"]}',fontweight='bold')
plt.tight_layout()
savefig(fig,'A_waveform_normalization.png'); plt.show()


---
## Phần B – Phân tích miền thời gian

In [ ]:
# Short-Time Energy
def ste(x,fs,fms=20,hms=10):
    fn,hn=int(fms*1e-3*fs),int(hms*1e-3*fs)
    tv,ev=[],[]
    s=0
    while s+fn<=len(x):
        f=x[s:s+fn]; ev.append(float(np.sum(f**2))/fn)
        tv.append((s+fn/2)/fs); s+=hn
    return np.array(tv),np.array(ev)

def stats(seg,lbl=''):
    pk=float(np.max(np.abs(seg))); rms=float(np.sqrt(np.mean(seg**2)))
    E=float(np.sum(seg**2)); cr=pk/(rms+1e-15)
    zcr=float(np.mean(np.abs(np.diff(np.sign(seg)))>0))
    nc=int(np.sum(np.abs(seg)>=0.999))
    if lbl: print(f'  [{lbl}]  Peak={pk:.5f}({db(pk):.1f}dBFS)  RMS={rms:.5f}({db(rms):.1f}dBFS)'
                  f'  E={E:.2f}  Crest={20*np.log10(cr+1e-15):.1f}dB  ZCR={zcr:.4f}  Clip={nc}')
    return dict(peak=pk,rms=rms,E=E,crest_db=20*np.log10(cr+1e-15),zcr=zcr,n_clip=nc,
                peak_dBFS=db(pk),rms_dBFS=db(rms))

st_full=stats(x,f'Toàn tệp ({dur:.2f}s)')
seg1=x[:int(0.5*fs)]; seg2=x[int(dur*0.4*fs):int((dur*0.4+0.5)*fs)]
s1=stats(seg1,'Đoạn đầu 0–0.5s'); s2=stats(seg2,f'Đoạn giữa {dur*0.4:.2f}–{dur*0.4+0.5:.2f}s')

# Bảng so sánh
print('\n  ┌─────────────────┬──────────────────┬──────────────────┐')
print(f'  │ Chỉ số          │ Đoạn đầu         │ Đoạn giữa        │')
print('  ├─────────────────┼──────────────────┼──────────────────┤')
for k,lab in [('peak_dBFS','Peak (dBFS)'),('rms_dBFS','RMS (dBFS)'),('E','Energy'),
              ('crest_db','Crest (dB)'),('zcr','ZCR'),('n_clip','Clipping')]:
    v1=f'{s1[k]:.4f}' if isinstance(s1[k],float) else str(s1[k])
    v2=f'{s2[k]:.4f}' if isinstance(s2[k],float) else str(s2[k])
    print(f'  │ {lab:<15}  │ {v1:>16} │ {v2:>16} │')
print('  └─────────────────┴──────────────────┴──────────────────┘')


In [ ]:
# Vẽ waveform + STE + 2 đoạn so sánh
t_ax=np.arange(len(x))/fs; t_ste,ste_v=ste(x,fs)
ste_n=ste_v/(np.max(ste_v)+1e-15)

fig=plt.figure(figsize=(13,9))
gs=gridspec.GridSpec(3,2,figure=fig,hspace=0.45,wspace=0.3)

ax1=fig.add_subplot(gs[0,:])
ax1.plot(t_ax,x,color='steelblue',lw=0.3,alpha=0.85)
ax1.axhline(st_full['rms'],color='orange',lw=1,ls='-.',label=f"RMS={st_full['rms']:.4f} ({st_full['rms_dBFS']:.1f}dBFS)")
ax1.axhline(-st_full['rms'],color='orange',lw=1,ls='-.')
ax1.axhline(st_full['peak'],color='purple',lw=0.7,ls=':',label=f"Peak={st_full['peak']:.4f}")
ax1.axhline(-st_full['peak'],color='purple',lw=0.7,ls=':')
ax1.axhline(1,color='red',lw=0.6,ls='--',alpha=0.5,label='Clipping ±1')
ax1.axhline(-1,color='red',lw=0.6,ls='--',alpha=0.5)
ax1.set_title(f'Phần B – Waveform toàn tệp: {info["filename"]}',fontweight='bold',fontsize=10)
ax1.set_ylabel('Biên độ',fontsize=8); ax1.set_ylim([-1.15,1.15])
ax1.legend(fontsize=7,loc='upper right'); ax1.grid(True,alpha=.3)

ax2=fig.add_subplot(gs[1,:])
ax2.fill_between(t_ste,ste_n,alpha=0.55,color='darkorange')
ax2.plot(t_ste,ste_n,color='darkorange',lw=0.7)
ax2.set_title('Short-Time Energy (frame=20ms, hop=10ms)',fontsize=9,fontweight='bold')
ax2.set_ylabel('STE (norm)',fontsize=8); ax2.set_ylim([0,1.05])
ax2.set_xlabel('Thời gian (s)',fontsize=8); ax2.grid(True,alpha=.3)

# 2 đoạn so sánh
for col,seg,st,ts,te,clr,lbl in [
    (0,seg1,s1,0.0,0.5,'steelblue','Đoạn đầu'),
    (1,seg2,s2,dur*0.4,dur*0.4+0.5,'tomato','Đoạn giữa')]:
    ax=fig.add_subplot(gs[2,col])
    t_s=np.linspace(ts,te,len(seg))
    ax.plot(t_s,seg,color=clr,lw=0.5)
    ax.fill_between(t_s,seg,alpha=0.2,color=clr)
    ax.axhline(st['rms'],color='orange',lw=1,ls='-.',label=f"RMS={st['rms_dBFS']:.1f}dBFS")
    ax.axhline(-st['rms'],color='orange',lw=1,ls='-.')
    ax.set_title(f'{lbl} | E={st["E"]:.2f} Crest={st["crest_db"]:.1f}dB',fontsize=9,fontweight='bold')
    ax.set_xlabel('s',fontsize=8); ax.set_ylabel('Biên độ',fontsize=8)
    ax.set_ylim([-1.15,1.15]); ax.legend(fontsize=7); ax.grid(True,alpha=.3)

plt.suptitle('Phần B – Phân tích miền thời gian',fontweight='bold',fontsize=11)
savefig(fig,'B_time_domain.png'); plt.show()


---
## Phần C – Phân tích FFT

$$X[k]=\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N},\quad\Delta f=F_s/N_{FFT},\quad\text{True res}=F_s/N_{window}$$

In [ ]:
def fft_compute(seg,fs,nfft=None,win='hamming'):
    N=len(seg)
    if nfft is None: nfft=int(2**np.ceil(np.log2(N)))
    wf={'hamming':np.hamming,'hann':np.hanning,'rectangular':np.ones,'blackman':np.blackman}.get(win,np.hamming)(N)
    cg=float(np.sum(wf))/N
    X=np.fft.rfft(seg*wf,n=nfft); f=np.fft.rfftfreq(nfft,1/fs)
    m=np.abs(X)/(N*cg); m[1:-1]*=2
    return dict(f=f,mag=m,mag_dB=20*np.log10(m+1e-15),
                delta_f=fs/nfft,true_res=fs/N,nfft=nfft,N=N,win=win,cg=cg)

# Đoạn ổn định 1s ở 30%–80%
ts_c=dur*0.3; te_c=min(ts_c+1.0,dur*0.8)
seg_c=x[int(ts_c*fs):int(te_c*fs)]
print(f'Đoạn: {ts_c:.3f}s–{te_c:.3f}s  ({len(seg_c):,} mẫu)')
print(f'True resolution = Fs/N = {fs}/{len(seg_c)} = {fs/len(seg_c):.4f} Hz')

nfft_std=int(2**np.ceil(np.log2(max(len(seg_c),2048))))
r=fft_compute(seg_c,fs,nfft=nfft_std)
print(f'NFFT={nfft_std:,}  Δf={r["delta_f"]:.4f}Hz  cg={r["cg"]:.5f}')

# Tìm đỉnh
freq_max=min(8000,fs//2)
mask=r['f']<=freq_max; min_d=max(3,int(20/r['delta_f']))
pk_idx,_=find_peaks(r['mag_dB'],height=-60,distance=min_d,prominence=3)
top5=pk_idx[np.argsort(r['mag_dB'][pk_idx])[::-1][:5]]
print('\n  Top đỉnh phổ:')
for i,p in enumerate(top5,1):
    print(f'  {i}. f={r["f"][p]:.2f}Hz  {r["mag_dB"][p]:.1f}dBFS')


In [ ]:
# Vẽ phổ FFT
fig,axes=plt.subplots(2,1,figsize=(13,9))
axes[0].plot(r['f'][mask],r['mag'][mask],color='steelblue',lw=0.7)
axes[0].fill_between(r['f'][mask],r['mag'][mask],alpha=0.25,color='steelblue')
axes[0].set_title(f'Phổ biên độ tuyến tính | NFFT={nfft_std:,} | Δf={r["delta_f"]:.4f}Hz',fontweight='bold',fontsize=9)
axes[0].set_ylabel('|X(f)| (normalized)',fontsize=8); axes[0].grid(True,alpha=.3)

axes[1].plot(r['f'][mask],r['mag_dB'][mask],color='darkorange',lw=0.8)
axes[1].set_title(f'Phổ dB | Hamming | True resolution={r["true_res"]:.3f}Hz (Fs/N_window)',fontweight='bold',fontsize=9)
axes[1].set_ylabel('Biên độ (dBFS)',fontsize=8); axes[1].set_xlabel('Tần số (Hz)',fontsize=8)
axes[1].set_ylim([-90,5]); axes[1].set_xlim([0,freq_max]); axes[1].grid(True,alpha=.3)

clrs_p=['red','limegreen','blueviolet','brown','deeppink']
for i,p in enumerate(top5):
    if r['f'][p]<=freq_max:
        c=clrs_p[i%5]
        axes[1].axvline(r['f'][p],color=c,lw=0.8,ls='--',alpha=0.7)
        axes[1].annotate(f'#{i+1}\n{r["f"][p]:.1f}Hz\n{r["mag_dB"][p]:.1f}dB',
            xy=(r['f'][p],r['mag_dB'][p]),xytext=(r['f'][p]+freq_max*0.012,r['mag_dB'][p]-8),
            fontsize=7,color=c,arrowprops=dict(arrowstyle='->',color=c,lw=0.5))
axes[0].set_xlim([0,freq_max])
fig.suptitle(f'Phần C – FFT: {info["filename"]}',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'C_fft.png'); plt.show()


In [ ]:
# So sánh NFFT nhỏ vs lớn (cùng frame)
nfft_s=max(256,int(2**np.floor(np.log2(len(seg_c))))//2)
nfft_l=min(262144,nfft_std*8)
rs=fft_compute(seg_c,fs,nfft=nfft_s); rl=fft_compute(seg_c,fs,nfft=nfft_l)

fig,axes=plt.subplots(2,2,figsize=(14,8))
for col,(rv,nf,c1,c2) in enumerate([(rs,nfft_s,'steelblue','darkorange'),(rl,nfft_l,'seagreen','tomato')]):
    mk=rv['f']<=freq_max
    axes[0,col].plot(rv['f'][mk],rv['mag'][mk],color=c1,lw=0.6)
    axes[0,col].set_title(f'NFFT={nf:,} | Δf={rv["delta_f"]:.3f}Hz (tuyến tính)',fontsize=9,fontweight='bold')
    axes[0,col].set_ylabel('|X(f)|',fontsize=8); axes[0,col].grid(True,alpha=.3)
    axes[1,col].plot(rv['f'][mk],rv['mag_dB'][mk],color=c2,lw=0.7)
    axes[1,col].set_title(f'NFFT={nf:,} | Δf={rv["delta_f"]:.3f}Hz (dB)',fontsize=9,fontweight='bold')
    axes[1,col].set_ylabel('dBFS',fontsize=8); axes[1,col].set_xlabel('Hz',fontsize=8)
    axes[1,col].set_ylim([-90,5]); axes[1,col].grid(True,alpha=.3)
    for ax in axes[:,col]: ax.set_xlim([0,freq_max])

true_res=fs/len(seg_c)
fig.suptitle(f'Phần C – So sánh NFFT: {nfft_s:,} vs {nfft_l:,}\nTrue resolution = Fs/N = {true_res:.3f}Hz (không đổi khi tăng NFFT)',
             fontweight='bold',fontsize=10)
plt.tight_layout(); savefig(fig,'C_compare_nfft.png'); plt.show()
print(f'⚠ Zero-padding KHÔNG tăng true resolution. Chỉ nội suy trục tần số.')


---
## Phần D – STFT và Spectrogram

$$X[m,k]=\sum_n x[n]\,w[n-mH]\,e^{-j2\pi kn/N_{FFT}}$$

In [ ]:
def stft(sig,fs,fms=25,hms=10,nfft=None,win='hamming'):
    npsg=round(fms*1e-3*fs); hn=round(hms*1e-3*fs)
    nov=max(0,min(npsg-hn,npsg-1))
    if nfft is None: nfft=int(2**np.ceil(np.log2(max(npsg,32))))
    f,t,S=_spectrogram(sig,fs=fs,window=win,nperseg=npsg,noverlap=nov,nfft=nfft,
                       scaling='spectrum',mode='magnitude')
    return dict(f=f,t=t,S=S,S_dB=20*np.log10(np.maximum(S,1e-10)),
                npsg=npsg,hn=hn,nov=nov,nfft=nfft,
                df=fs/nfft,true_df=fs/npsg,dt_ms=hn/fs*1000,
                ovl=nov/npsg*100,fms=fms,hms=hms)

ts_d=dur*0.1; te_d=min(ts_d+10,dur)
seg_d=x[int(ts_d*fs):int(te_d*fs)]
freq_max_d=float(min(8000,fs//2))

sg=stft(seg_d,fs,fms=25,hms=10)
print(f'Spectrogram 25ms/10ms: {sg["npsg"]}mẫu/frame | Δf={sg["df"]:.2f}Hz | Δt={sg["dt_ms"]:.1f}ms | '
      f'overlap={sg["ovl"]:.0f}% | shape={sg["S_dB"].shape}')


In [ ]:
# Vẽ spectrogram chuẩn
fmask=sg['f']<=freq_max_d
fig,ax=plt.subplots(figsize=(13,5))
im=ax.pcolormesh(sg['t'],sg['f'][fmask],sg['S_dB'][fmask,:],
                 shading='gouraud',cmap='inferno',vmin=-90,vmax=0)
cb=plt.colorbar(im,ax=ax,pad=0.01); cb.set_label('Biên độ (dB)',fontsize=8)
ax.set_xlabel('Thời gian (s)',fontsize=8); ax.set_ylabel('Tần số (Hz)',fontsize=8)
ax.set_title(f'Phần D – Spectrogram 25ms/10ms | Δf={sg["df"]:.2f}Hz | Δt={sg["dt_ms"]:.1f}ms | {info["filename"]}',
             fontweight='bold',fontsize=9)
ax.set_ylim([0,freq_max_d])
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_: f'{v/1000:.1f}kHz' if v>=1000 else f'{int(v)}Hz'))
plt.tight_layout(); savefig(fig,'D_spectrogram.png'); plt.show()


In [ ]:
# So sánh 3 frame length: 10/25/50ms
fig,axes=plt.subplots(3,1,figsize=(13,12),sharex=True)
summary=[]
for ax,fl in zip(axes,[10.0,25.0,50.0]):
    sg_i=stft(seg_d,fs,fms=fl,hms=10)
    fmask_i=sg_i['f']<=freq_max_d
    im=ax.pcolormesh(sg_i['t'],sg_i['f'][fmask_i],sg_i['S_dB'][fmask_i,:],
                     shading='gouraud',cmap='magma',vmin=-90,vmax=0)
    plt.colorbar(im,ax=ax,label='dB',pad=0.01).ax.tick_params(labelsize=7)
    ax.set_ylabel('Tần số (Hz)',fontsize=8)
    ax.set_title(f'Frame={fl:.0f}ms | Δt={sg_i["dt_ms"]:.1f}ms | True Δf={sg_i["true_df"]:.2f}Hz | NFFT Δf={sg_i["df"]:.2f}Hz',
                 fontsize=9,fontweight='bold')
    ax.set_ylim([0,freq_max_d])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_: f'{int(v/1000)}kHz' if v>=1000 else f'{int(v)}Hz'))
    summary.append((fl,sg_i['dt_ms'],sg_i['true_df'],sg_i['df']))
axes[-1].set_xlabel('Thời gian (s)',fontsize=8)
fig.suptitle('Phần D – Trade-off Time/Frequency Resolution (Hop=10ms cố định)',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'D_compare_frames.png'); plt.show()

print('\n  Frame(ms) | Δt(ms) | True Δf(Hz) | NFFT Δf(Hz)')
print('  ----------+--------+-------------+------------')
for fl,dt,tdf,ndf in summary:
    print(f'  {fl:>8.0f}  | {dt:>5.1f}  | {tdf:>10.3f}  | {ndf:>10.3f}')
print('  → Frame ngắn: Δt tốt → rõ transient')
print('  → Frame dài : Δf tốt → rõ harmonic series')


---
## Phần E – Thí nghiệm cửa sổ

In [ ]:
# Đặc tính 4 cửa sổ
N_w=1024; nfft_w=8192
wins={'Rectangular':np.ones(N_w),'Hamming':np.hamming(N_w),'Hann':np.hanning(N_w),'Blackman':np.blackman(N_w)}
clrs_w=['steelblue','darkorange','seagreen','tomato']

fig,axes=plt.subplots(2,4,figsize=(16,7))
print(f'  {"Cửa sổ":<12} {"CG":>8} {"Main-lobe":>12} {"Side-lobe(dB)":>15}')
print('  '+'-'*50)
for col,(wname,w,clr) in enumerate(zip(wins,wins.values(),clrs_w)):
    cg=float(np.mean(w)); tn=np.arange(N_w)
    W=np.fft.rfft(w/N_w,n=nfft_w); Wdb=20*np.log10(np.abs(W)+1e-15)
    pk=int(np.argmax(Wdb)); pkv=float(Wdb[pk])
    lo,hi=pk,pk
    while lo>0 and Wdb[lo]>pkv-3: lo-=1
    while hi<len(Wdb)-1 and Wdb[hi]>pkv-3: hi+=1
    ml=hi-lo
    sl=Wdb.copy(); sl[max(0,lo-5):hi+5]=-200
    print(f'  {wname:<12} {cg:>8.4f} {ml:>12}bins {float(np.max(sl))-pkv:>13.1f}dB')
    axes[0,col].plot(tn,w,color=clr,lw=1); axes[0,col].set_title(wname,fontweight='bold',fontsize=9)
    axes[0,col].set_ylim([-0.1,1.15]); axes[0,col].set_xlabel('Mẫu n',fontsize=7)
    if col==0: axes[0,col].set_ylabel('w[n]',fontsize=7)
    axes[0,col].grid(True,alpha=.3)
    fd=np.arange(len(Wdb))[:nfft_w//32]
    axes[1,col].plot(fd,Wdb[:nfft_w//32],color=clr,lw=0.8)
    axes[1,col].axhline(-3,color='gray',lw=0.6,ls='--'); axes[1,col].set_ylim([-100,5])
    axes[1,col].set_xlabel('Bin (norm.)',fontsize=7)
    if col==0: axes[1,col].set_ylabel('|W|(dB)',fontsize=7)
    axes[1,col].grid(True,alpha=.3)
fig.suptitle(f'Phần E – Hình dạng và đáp ứng tần số cửa sổ (N={N_w})',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'E_window_overview.png'); plt.show()


In [ ]:
# So sánh Rectangular vs Hamming trên frame thực
N_fr=round(25e-3*fs); i0_e=int(dur*0.4*fs)
frame_e=x[i0_e:i0_e+N_fr]
if len(frame_e)<N_fr: frame_e=np.concatenate([frame_e,np.zeros(N_fr-len(frame_e))])
nfft_e=int(2**np.ceil(np.log2(N_fr)))*4; freq_max_e=float(min(6000,fs//2))

res_e={}
for wname,w in [('Rectangular',np.ones(N_fr)),('Hamming',np.hamming(N_fr))]:
    cg=float(np.mean(w)); X=np.fft.rfft(frame_e*w,n=nfft_e)
    m=np.abs(X)/(N_fr*cg); m[1:-1]*=2
    res_e[wname]=dict(f=np.fft.rfftfreq(nfft_e,1/fs),xw=frame_e*w,
                       mag_dB=20*np.log10(m+1e-15),w=w)

t_fr=np.arange(N_fr)/fs*1000
fig=plt.figure(figsize=(14,11))
gs_e=gridspec.GridSpec(3,2,figure=fig,hspace=0.45,wspace=0.3)
clrs_e={'Rectangular':'steelblue','Hamming':'darkorange'}

for col,(wn,res) in enumerate(res_e.items()):
    c=clrs_e[wn]; mk=res['f']<=freq_max_e
    ax=fig.add_subplot(gs_e[0,col])
    ax.plot(t_fr,res['xw'],color=c,lw=0.6)
    ax.plot(t_fr,res['w']*np.max(np.abs(frame_e)),color='gray',lw=0.7,ls='--',alpha=0.5,label='Window (scaled)')
    ax.set_title(f'Frame × {wn}',fontsize=9,fontweight='bold')
    ax.set_xlabel('ms',fontsize=7); ax.legend(fontsize=7); ax.grid(True,alpha=.3)
    ax=fig.add_subplot(gs_e[1,col])
    ax.plot(res['f'][mk],res['mag_dB'][mk],color=c,lw=0.7)
    pi=int(np.argmax(res['mag_dB'][mk])); sl=res['mag_dB'][mk].copy()
    sl[max(0,pi-10):pi+10]=-200; slv=float(np.max(sl))
    ax.axhline(slv,color='red',lw=0.6,ls='--',label=f'Side-lobe≈{slv:.1f}dB')
    ax.set_title(f'Phổ dB – {wn}',fontsize=9,fontweight='bold')
    ax.set_ylim([-90,5]); ax.set_xlabel('Hz',fontsize=7); ax.legend(fontsize=7); ax.grid(True,alpha=.3)

ax_ov=fig.add_subplot(gs_e[2,0])
for wn,res in res_e.items():
    mk=res['f']<=freq_max_e
    ax_ov.plot(res['f'][mk],res['mag_dB'][mk],color=clrs_e[wn],lw=0.7,alpha=0.85,label=wn)
ax_ov.set_ylim([-90,5]); ax_ov.set_title('Chồng phổ',fontsize=9,fontweight='bold')
ax_ov.set_xlabel('Hz',fontsize=7); ax_ov.legend(fontsize=8); ax_ov.grid(True,alpha=.3)

ax_df=fig.add_subplot(gs_e[2,1])
fr_r=res_e['Rectangular']['f']; mk=fr_r<=freq_max_e
diff=res_e['Rectangular']['mag_dB'][mk]-res_e['Hamming']['mag_dB'][mk]
ax_df.fill_between(fr_r[mk],diff,0,where=diff>0,color='steelblue',alpha=0.5,label='Rect>Ham')
ax_df.fill_between(fr_r[mk],diff,0,where=diff<0,color='darkorange',alpha=0.5,label='Ham>Rect')
ax_df.axhline(0,color='black',lw=0.5); ax_df.set_title('Chênh lệch Rect−Ham (dB)',fontsize=9,fontweight='bold')
ax_df.set_xlabel('Hz',fontsize=7); ax_df.legend(fontsize=8); ax_df.grid(True,alpha=.3)

fig.suptitle(f'Phần E – So sánh cửa sổ trên frame thực | N={N_fr} | NFFT={nfft_e:,} | {info["filename"]}',
             fontweight='bold',fontsize=10)
savefig(fig,'E_window_comparison.png'); plt.show()
print('  Nhận xét: Hamming side-lobe thấp hơn → ít leakage; Main-lobe rộng hơn → khó tách tần số gần nhau')


---
## Phần F – Lọc số FIR

$$y[n]=\sum_{r=0}^{M}b_r x[n-r],\quad\tau_{group}=\frac{M}{2}\text{ mẫu}$$

In [ ]:
nyq_f=fs/2.0; TAPS=201
lp_cut=min(2000,int(nyq_f*0.40))
hp_cut=min(3000,int(nyq_f*0.55))
bp_lo =min(500, int(nyq_f*0.08))
bp_hi =min(3500,int(nyq_f*0.65))
gd_ms=(TAPS-1)/2/fs*1000

b_lp=firwin(TAPS,lp_cut,         pass_zero=True, fs=fs,window='hamming')
b_hp=firwin(TAPS,hp_cut,         pass_zero=False,fs=fs,window='hamming')
b_bp=firwin(TAPS,[bp_lo,bp_hi],  pass_zero=False,fs=fs,window='hamming')

print(f'Fs={fs}Hz | Nyquist={nyq_f:.0f}Hz | Taps={TAPS} | Group delay={gd_ms:.3f}ms')
for b,lbl in [(b_lp,f'LP {lp_cut}Hz'),(b_hp,f'HP {hp_cut}Hz'),(b_bp,f'BP {bp_lo}-{bp_hi}Hz')]:
    print(f'  {lbl}: symmetric={np.allclose(b,b[::-1])}  b[0..3]={b[:4].round(6)}')


In [ ]:
# Vẽ frequency response: magnitude + phase + group delay
filter_cfgs=[(b_lp,f'Low-pass {lp_cut}Hz','steelblue'),
             (b_hp,f'High-pass {hp_cut}Hz','tomato'),
             (b_bp,f'Band-pass {bp_lo}-{bp_hi}Hz','seagreen')]
fig,axes=plt.subplots(3,3,figsize=(16,11))
for col,(b,lbl,clr) in enumerate(filter_cfgs):
    w_fr,H=freqz(b,worN=8192,fs=fs)
    H_dB=20*np.log10(np.maximum(np.abs(H),1e-15))
    H_ph=np.unwrap(np.angle(H))*180/np.pi
    w_gd,gd_s=group_delay((b,[1.0]),w=8192,fs=fs); gd_m=gd_s/fs*1000
    # Magnitude
    axes[0,col].plot(w_fr,H_dB,color=clr,lw=1.2)
    axes[0,col].axhline(-3,color='black',lw=0.7,ls='--',label='-3dB')
    axes[0,col].axhline(-40,color='gray',lw=0.6,ls=':',label='-40dB')
    idx3=np.where(H_dB<=-3)[0]
    if len(idx3)>0:
        axes[0,col].axvline(w_fr[idx3[0]],color='red',lw=0.8,ls='-.')
        axes[0,col].text(w_fr[idx3[0]]+fs*.005,-15,f'{w_fr[idx3[0]]:.0f}Hz\n-3dB',fontsize=7,color='red')
    axes[0,col].set_title(f'{lbl}\n{TAPS} taps',fontsize=9,fontweight='bold')
    axes[0,col].set_ylim([-90,5]); axes[0,col].legend(fontsize=7); axes[0,col].grid(True,alpha=.3)
    # Phase
    axes[1,col].plot(w_fr,H_ph,color=clr,lw=0.9)
    axes[1,col].set_title(f'Phase – {lbl}',fontsize=9,fontweight='bold')
    axes[1,col].set_ylabel('Phase (°)',fontsize=7); axes[1,col].grid(True,alpha=.3)
    # Group delay
    axes[2,col].plot(w_gd,gd_m,color=clr,lw=0.9)
    axes[2,col].axhline(gd_ms,color='red',lw=0.8,ls='--',label=f'Nominal {gd_ms:.2f}ms')
    axes[2,col].set_title(f'Group delay – {lbl}',fontsize=9,fontweight='bold')
    axes[2,col].set_xlabel('Tần số (Hz)',fontsize=7); axes[2,col].set_ylabel('ms',fontsize=7)
    axes[2,col].legend(fontsize=7); axes[2,col].grid(True,alpha=.3)
fig.suptitle(f'Phần F – Đáp ứng tần số FIR | {info["filename"]}',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'F_filter_response.png'); plt.show()


In [ ]:
# Áp dụng filter + bù group delay + vẽ phổ trước/sau
def apply_fir(sig,b):
    d=(len(b)-1)//2; y=lfilter(b,[1.0],sig)
    out=np.empty_like(sig); out[:len(sig)-d]=y[d:]; out[len(sig)-d:]=0
    return out

def spec_db(sig,fs,nf=None):
    n=min(len(sig),fs); nf2=int(2**np.ceil(np.log2(n)))*2 if nf is None else nf
    w=np.hamming(n); cg=float(np.mean(w))
    m=np.abs(np.fft.rfft(sig[:n]*w,n=nf2))/(n*cg); m[1:-1]*=2
    return np.fft.rfftfreq(nf2,1/fs),20*np.log10(m+1e-15)

y_lp=apply_fir(x,b_lp); y_hp=apply_fir(x,b_hp); y_bp=apply_fir(x,b_bp)

fig,axes=plt.subplots(3,3,figsize=(16,11))
for col,(y,b,lbl,suf) in enumerate([
        (y_lp,b_lp,f'LP {lp_cut}Hz','lowpass'),
        (y_hp,b_hp,f'HP {hp_cut}Hz','highpass'),
        (y_bp,b_bp,f'BP {bp_lo}-{bp_hi}Hz','bandpass')]):
    fx,Xdb=spec_db(x,fs); fy,Ydb=spec_db(y,fs)
    mk=fx<=freq_max
    # Phổ chồng
    axes[0,col].plot(fx[mk],Xdb[mk],color='steelblue',lw=0.8,alpha=0.85,label='Trước')
    axes[0,col].plot(fy[mk],Ydb[mk],color='darkorange',lw=0.8,alpha=0.9,label=f'Sau {lbl}')
    axes[0,col].set_ylim([-90,5]); axes[0,col].legend(fontsize=7)
    axes[0,col].set_title(f'Phổ trước/sau – {lbl}',fontsize=9,fontweight='bold')
    axes[0,col].grid(True,alpha=.3)
    # Attenuation
    atten=Xdb[mk]-Ydb[mk]
    axes[1,col].plot(fx[mk],atten,color='seagreen',lw=0.8)
    axes[1,col].axhline(3,color='red',lw=0.7,ls='--',label='3dB')
    axes[1,col].axhline(40,color='gray',lw=0.6,ls=':',label='40dB')
    axes[1,col].set_title(f'Suy giảm (dB)',fontsize=9,fontweight='bold')
    axes[1,col].legend(fontsize=7); axes[1,col].grid(True,alpha=.3)
    # |H(f)|
    w_fr2,H2=freqz(b,worN=len(fx),fs=fs)
    H_dB2=20*np.log10(np.maximum(np.abs(H2),1e-15)); mkh=w_fr2<=freq_max
    axes[2,col].plot(w_fr2[mkh],H_dB2[mkh],color='tomato',lw=1,label='|H(f)|')
    axes[2,col].axhline(-3,color='black',lw=0.7,ls='--',label='-3dB')
    axes[2,col].set_ylim([-90,5]); axes[2,col].legend(fontsize=7)
    axes[2,col].set_title(f'|H(f)| – {lbl}',fontsize=9,fontweight='bold')
    axes[2,col].set_xlabel('Tần số (Hz)',fontsize=7); axes[2,col].grid(True,alpha=.3)
    for ax in axes[:,col]: ax.set_xlim([0,freq_max])
    # Lưu WAV
    p=save_wav(y,fs,f'{prefix}_{suf}_{lp_cut if suf=="lowpass" else (hp_cut if suf=="highpass" else f"{bp_lo}_{bp_hi}") }hz.wav')
    print(f'  → {p}')

fig.suptitle(f'Phần F – FIR Filtering | {info["filename"]}',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'F_before_after.png'); plt.show()


---
## Phần G – Lượng tử hóa, Resampling và Mã hóa

$$\text{SNR}_Q=6.02B+4.77-20\log_{10}(X_{max}/\sigma_x)$$
$$R_{PCM}=F_s\times B\times C$$

In [ ]:
# G1: Lượng tử hóa
def quant(sig,B):
    L=2**B; d=2/L; xc=np.clip(sig,-1,1-d)
    return np.clip(np.round(xc/d)*d,-1,1-d)

def snr_meas(sig,sq):
    e=sq-sig; sp=np.mean(sig**2); np_=np.mean(e**2)
    return float(10*np.log10(sp/np_)) if np_>1e-30 else float('inf')

def snr_theory(B,sig):
    sx=float(np.std(sig))
    return 6.02*B+4.77-20*np.log10(1/sx) if sx>1e-15 else 0

BITS=[4,6,8,12,16]; qr={}
print(f'  RMS={np.sqrt(np.mean(x**2)):.6f} ({db(np.sqrt(np.mean(x**2))):.2f}dBFS)\n')
print(f'  {"B":>4} {"L":>8} {"Δ":>12} {"SNR đo":>10} {"SNR lý thuyết":>14}')
print('  '+'-'*54)
for B in BITS:
    xq=quant(x,B); sm=snr_meas(x,xq); st=snr_theory(B,x); d_=2/2**B
    qr[B]=dict(xq=xq,snr_m=sm,snr_t=st,L=2**B,delta=d_)
    print(f'  {B:>4} {2**B:>8,} {d_:>12.7f} {sm:>10.2f}dB {st:>12.2f}dB')


In [ ]:
# Vẽ SNR + nhiễu lượng tử
fig,axes=plt.subplots(1,2,figsize=(14,6))
snr_m=[qr[B]['snr_m'] for B in BITS]; snr_t=[qr[B]['snr_t'] for B in BITS]
axes[0].plot(BITS,snr_m,'o-',color='steelblue',lw=1.5,ms=7,label='SNR đo')
axes[0].plot(BITS,snr_t,'s--',color='tomato',lw=1.2,ms=7,label='SNR lý thuyết')
axes[0].plot(BITS,[6.02*B for B in BITS],':k',lw=0.9,alpha=0.4,label='6.02dB/bit')
for B,sm in zip(BITS,snr_m):
    if sm!=float('inf'): axes[0].annotate(f'{sm:.1f}',xy=(B,sm),xytext=(B+.3,sm+1.5),fontsize=8,color='steelblue')
axes[0].set_xlabel('Bit/mẫu (B)',fontsize=9); axes[0].set_ylabel('SNR (dB)',fontsize=9)
axes[0].set_title('SNR lượng tử theo số bit',fontweight='bold'); axes[0].legend(fontsize=9)
axes[0].set_xticks(BITS); axes[0].grid(True,alpha=.3)

ns=min(int(0.05*fs),len(x)); tmx=np.arange(ns)/fs*1000
for B,c in zip([BITS[0],BITS[len(BITS)//2],BITS[-1]],['tomato','darkorange','steelblue']):
    noise=qr[B]['xq'][:ns]-x[:ns]; d_=qr[B]['delta']
    axes[1].plot(tmx,noise,color=c,lw=0.5,alpha=0.8,label=f'B={B} Δ={d_:.5f}')
    axes[1].axhline(d_/2,color=c,lw=0.4,ls='--',alpha=0.4); axes[1].axhline(-d_/2,color=c,lw=0.4,ls='--',alpha=0.4)
axes[1].set_xlabel('ms',fontsize=9); axes[1].set_ylabel('e[n]=x̂[n]−x[n]',fontsize=9)
axes[1].set_title('Nhiễu lượng tử (50ms đầu)',fontweight='bold'); axes[1].legend(fontsize=8); axes[1].grid(True,alpha=.3)
fig.suptitle('Phần G – Lượng tử hóa PCM',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'G_quantization.png'); plt.show()

for B in [4,8,16]:
    p=save_wav(qr[B]['xq'],fs,f'{prefix}_{B}bit.wav'); print(f'  → {p}')


In [ ]:
# G2: Resampling
targets=[16000,8000] if fs>8000 else []
resampled={}
for ft in targets:
    g=gcd(fs,ft); xr=resample_poly(x,ft//g,fs//g).astype(np.float64)
    xr=np.clip(xr,-1,1); resampled[ft]=xr
    p=save_wav(xr,ft,f'{prefix}_resample_{ft}hz.wav'); print(f'  → {p}  ({len(xr):,} mẫu)')

# Vẽ phổ so sánh
all_s={fs:x,**resampled}
fig,axes=plt.subplots(len(all_s),2,figsize=(14,3.5*len(all_s)))
if len(all_s)==1: axes=axes.reshape(1,-1)
clrs_r=['steelblue','darkorange','seagreen']
for row,(fc,sg) in enumerate(all_s.items()):
    c=clrs_r[row%3]; ns2=min(len(sg),int(0.5*fc)); tax=np.arange(ns2)/fc
    axes[row,0].plot(tax,sg[:ns2],color=c,lw=0.4)
    axes[row,0].set_title(f'Waveform Fs={fc:,}Hz Nyquist={fc//2:,}Hz',fontsize=9,fontweight='bold')
    axes[row,0].set_xlabel('s',fontsize=8); axes[row,0].set_ylabel('Biên độ',fontsize=8); axes[row,0].grid(True,alpha=.3)
    ns3=min(len(sg),fc); nf3=int(2**np.ceil(np.log2(ns3)))
    Xdb3=20*np.log10(np.maximum(np.abs(np.fft.rfft(sg[:ns3]*np.hamming(ns3),n=nf3)),1e-15))
    fa3=np.fft.rfftfreq(nf3,1/fc); mk3=fa3<=min(freq_max,fc//2)
    axes[row,1].plot(fa3[mk3],Xdb3[mk3],color=c,lw=0.6)
    axes[row,1].axvline(fc//2,color='red',lw=0.8,ls='--',label=f'Nyquist={fc//2:,}Hz')
    axes[row,1].set_ylim([-90,5]); axes[row,1].legend(fontsize=8); axes[row,1].grid(True,alpha=.3)
    axes[row,1].set_title(f'Phổ Fs={fc:,}Hz',fontsize=9,fontweight='bold')
    axes[row,1].set_xlabel('Hz',fontsize=8); axes[row,1].set_ylabel('dBFS',fontsize=8)
fig.suptitle('Phần G – Resampling (resample_poly, anti-alias tự động)',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'G_resampling.png'); plt.show()


In [ ]:
# G3: Bit rate và compression ratio lý thuyết
dur_g=info['duration']; ch_g=info['channels']; fsz_g=info['file_size']
pcm_ref=44100*16*2
cfgs=[('PCM 8b mono 8kHz',8000,8,1),('PCM 8b mono 16kHz',16000,8,1),
      ('PCM 16b mono 16kHz',16000,16,1),('PCM 16b mono 44.1kHz',44100,16,1),
      ('PCM 16b stereo 44.1kHz',44100,16,2),(f'Gốc ({fs}Hz {ch_g}ch)',fs,16,ch_g)]
print(f'  {"Cấu hình":<30} {"kbps":>8} {"MB":>8} {"Ratio":>10} {"Saving":>8}')
print('  '+'-'*66)
names_g=[]; kbps_g=[]
for nm,fsc,b_,c_ in cfgs:
    bps=fsc*b_*c_; sz=bps*dur_g/8/1024/1024; rat=pcm_ref/bps; sv=(1-bps/pcm_ref)*100
    print(f'  {nm:<30} {bps/1000:>8.1f} {sz:>8.3f} {rat:>9.2f}:1 {sv:>7.1f}%')
    names_g.append(nm.split('(')[0].strip()); kbps_g.append(bps/1000)
act_kbps=fsz_g*8/dur_g/1000; act_mb=fsz_g/1024/1024; act_rat=pcm_ref/(act_kbps*1000)
print(f'  {"File thực tế":<30} {act_kbps:>8.1f} {act_mb:>8.3f} {act_rat:>9.2f}:1 {(1-act_kbps*1000/pcm_ref)*100:>7.1f}%')
names_g.append('File thực tế'); kbps_g.append(act_kbps)

fig,ax=plt.subplots(figsize=(12,5))
bars=ax.bar(range(len(names_g)),kbps_g,
            color=plt.cm.RdYlGn(np.linspace(0.8,0.2,len(names_g))),edgecolor='black',lw=0.5)
for bar,k in zip(bars,kbps_g):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+5,f'{k:.0f}',ha='center',va='bottom',fontsize=8)
ax.set_xticks(range(len(names_g))); ax.set_xticklabels(names_g,rotation=18,ha='right',fontsize=8)
ax.set_ylabel('Bit rate (kbps)',fontsize=9)
ax.set_title('Phần G – So sánh Bit Rate các cấu hình',fontweight='bold')
ax.set_ylim([0,max(kbps_g)*1.15]); ax.grid(True,axis='y',alpha=.3)
plt.tight_layout(); savefig(fig,'G_bitrate.png'); plt.show()


In [ ]:
# G4: Nén OGG/Vorbis thực tế
import soundfile as sf
sig_f32=x.astype(np.float32); dur_g=info['duration']
wav_bytes=len(x)*2; wav_kbps=fs*16/1000
print(f'  PCM 16-bit: {wav_bytes//1024}KB  ({wav_kbps:.1f}kbps)')
print(f'  {"Quality":>8}  {"Size(KB)":>9}  {"kbps":>8}  {"Ratio":>8}  {"Saving":>8}')
print('  '+'─'*52)
ogg_results={}
for q in [1,3,6,9]:
    ogg_path=os.path.join(AUDIO_DIR,f'{prefix}_ogg_q{q}.ogg')
    try:
        sf.write(ogg_path,sig_f32,fs,format='OGG',subtype='VORBIS')
        ob=os.path.getsize(ogg_path); okbps=ob*8/dur_g/1000
        ratio=wav_bytes/ob; saving=(1-ob/wav_bytes)*100
        print(f'  Q={q:>3}:  {ob//1024:>6}KB  {okbps:>8.1f}  {ratio:>7.2f}:1  {saving:>7.1f}%')
        ogg_results[q]=dict(size=ob,kbps=okbps,ratio=ratio,saving=saving)
    except Exception as e: print(f'  Q={q}: lỗi – {e}')

# Vẽ bar chart so sánh
if ogg_results:
    qs=sorted(ogg_results); sizes=[ogg_results[q]['size']//1024 for q in qs]
    ratios_=[ogg_results[q]['ratio'] for q in qs]
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    bars=axes[0].bar([str(q) for q in qs],sizes,
                     color=plt.cm.Blues(np.linspace(0.4,0.9,len(qs))),edgecolor='black',lw=0.5)
    axes[0].axhline(wav_bytes//1024,color='red',lw=1.5,ls='--',label=f'PCM={wav_bytes//1024}KB')
    for bar,s in zip(bars,sizes): axes[0].text(bar.get_x()+bar.get_width()/2,s+1,f'{s}KB',ha='center',fontsize=8)
    axes[0].set_xlabel('OGG Quality',fontsize=9); axes[0].set_ylabel('Kích thước (KB)',fontsize=9)
    axes[0].set_title('File Size: PCM vs OGG',fontweight='bold'); axes[0].legend(fontsize=8)
    axes[0].set_ylim([0,wav_bytes//1024*1.25]); axes[0].grid(True,axis='y',alpha=.3)
    axes[1].plot([str(q) for q in qs],ratios_,'o-',color='steelblue',lw=1.5,ms=7)
    axes[1].axhline(1,color='red',lw=0.8,ls='--',label='PCM ratio=1')
    for xi,(q,r_) in enumerate(zip(qs,ratios_)): axes[1].annotate(f'{r_:.1f}:1',xy=(xi,r_),xytext=(xi+.05,r_+max(ratios_)*.03),fontsize=8,color='steelblue')
    axes[1].set_xlabel('OGG Quality',fontsize=9); axes[1].set_ylabel('Compression Ratio',fontsize=9)
    axes[1].set_title('Compression Ratio OGG vs PCM',fontweight='bold'); axes[1].legend(fontsize=8); axes[1].grid(True,alpha=.3)
    fig.suptitle(f'Phần G – PCM vs OGG/Vorbis | {info["filename"]}',fontweight='bold',fontsize=11)
    plt.tight_layout(); savefig(fig,'G_compression_ogg.png'); plt.show()
    print('  ⚠ OGG là LOSSY: không phục hồi thông tin đã mất khi giải nén')


---
## Tổng kết kết quả

In [ ]:
print('='*62)
print('  KẾT QUẢ ĐẦU RA – CSE457 Lab 1')
print('  Sinh viên: Dương Tiến Đạt – MSSV: 2251172268')
print('='*62)
print(f'\n  📁 audio/  ({len([f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav") or f.endswith(".ogg")])} files)')
for f in sorted(os.listdir(AUDIO_DIR)):
    sz=os.path.getsize(os.path.join(AUDIO_DIR,f))
    print(f'     {f:<50} {sz//1024:>4}KB')
print(f'\n  📁 figures/  ({len([f for f in os.listdir(FIGURES_DIR) if f.endswith(".png")])} PNG)')
for f in sorted(os.listdir(FIGURES_DIR)):
    sz=os.path.getsize(os.path.join(FIGURES_DIR,f))
    print(f'     {f:<50} {sz//1024:>4}KB')


## Câu hỏi báo cáo (tóm tắt)

| # | Câu hỏi | Trả lời ngắn |
|---|---------|-------------|
| 1 | Tại sao Fs=44.1kHz chỉ đến 22.05kHz? | Định lý Nyquist: $F_{max} = F_s/2$ — tần số cao hơn gây aliasing |
| 2 | NFFT tăng 2048→8192, frame 25ms: thay đổi gì? | Δf nhỏ hơn (phổ mịn hơn) nhưng **true resolution = Fs/N_window không đổi** |
| 3 | Hamming giảm leakage nhưng khó tách đỉnh gần? | Side-lobe thấp ✓ nhưng main-lobe rộng ~2× → cần tần số cách nhau hơn |
| 4 | FIR 201 taps, 44.1kHz: delay bao nhiêu ms? | τ = (201−1)/2 / 44100 ≈ **2.27 ms** — OK cho VoIP, biên giới live sound |
| 5 | Tại sao giảm mức tín hiệu làm SNR_Q giảm? | σx giảm → 20log10(Xmax/σx) tăng → SNR_Q = 6.02B+4.77−... giảm |
| 6 | WAV 16-bit stereo 44.1kHz 60s = ? MB | (44100×16×2×60)/8 = **10.09 MB** vs MP3 128kbps = 0.96 MB |
| 7 | Nghe tốt hơn ≠ SNR lớn hơn (2 trường hợp) | (1) MP3 128kbps vs PCM 8-bit; (2) Tube amp THD vs solid-state |
